Maintenant que nous savons evoyer des données en CBOR, nous allons nous interesser à l'émission de série temporelles, c'est à dire pouvoir envoyer plusieurs valeurs en meme temps. Un des buts est de réduire le nombre de paquets envoyés et donc d'économiser de l'énergie si on n'a pas besoin des données en temps réel. En effet le récepteur ne recevra les données que quand elles seront toutes collectées.

Facile, vous allez me dire, il suffit de faire un tableau. On va voir.

Dans un premier temps, on relance notre programme serveur aui fait pas grand chose, mais affiche le CBOR reçu. Si ca coince c'est que vous avez oublié de couper le serveur dans un autre notebook. Allez au notebook précédent et tuez le processus.


In [7]:
#code server
import cbor2 as cbor
import asyncio 

try:
    if not server_transport.is_closing():
        server_transport.close()
        print("🟡 Ancien serveur arrêté proprement.")
except NameError:
    # La variable n'existe pas encore (premier lancement)
    print("⚪ Premier lancement : aucun serveur à arrêter.")
    
# 1. Définition du protocole Serveur
class EchoServerProtocol(asyncio.DatagramProtocol):
    def connection_made(self, transport):
        self.transport = transport
        print("🟢 Serveur Asyncio UDP : DÉMARRÉ (127.0.0.1:33033)")

    def datagram_received(self, data, addr):
        data = cbor.loads(data)
        for d in data:
            print(f"données: {d} %HR")

# 2. Fonction pour lancer le serveur en arrière-plan
async def start_background_server():
    loop = asyncio.get_running_loop()
    
    # create_datagram_endpoint crée le socket et l'attache à la boucle
    transport, protocol = await loop.create_datagram_endpoint(
        lambda: EchoServerProtocol(),
        local_addr=('127.0.0.1', 33033)
    )
    
    # On retourne le transport pour pouvoir le fermer plus tard si besoin
    return transport

# 3. Lancement de la tâche serveur
# On stocke 'server_transport' dans une variable globale pour pouvoir l'arrêter
server_transport = await start_background_server()

🟢 Serveur Asyncio UDP : DÉMARRÉ (127.0.0.1:33033)


données: 2905 %HR
données: 2726 %HR
données: 2562 %HR
données: 2536 %HR
données: 2252 %HR
données: 2421 %HR
données: 2369 %HR
données: 2138 %HR
données: 2094 %HR
données: 2232 %HR
données: 2000 %HR
données: 2000 %HR
données: 2000 %HR
données: 2000 %HR
données: 2000 %HR
données: 2000 %HR
données: 2000 %HR
données: 2163 %HR
données: 2125 %HR
données: 2173 %HR
données: 2137 %HR
données: 2014 %HR
données: 2163 %HR
données: 2114 %HR
données: 2319 %HR
données: 2350 %HR
données: 2261 %HR
données: 2183 %HR
données: 2034 %HR
données: 2000 %HR
données: 2000 %HR
données: 2000 %HR
données: 2000 %HR
données: 2000 %HR
données: 2000 %HR
données: 2000 %HR
données: 2000 %HR
données: 2178 %HR
données: 2305 %HR
données: 2525 %HR
données: 2697 %HR
données: 2757 %HR
données: 2575 %HR
données: 2564 %HR
données: 2626 %HR
données: 2782 %HR
données: 2832 %HR
données: 3070 %HR
données: 3018 %HR
données: 3186 %HR
données: 3375 %HR
données: 3202 %HR
données: 3403 %HR
données: 3576 %HR
données: 3653 %HR
données: 3

Bon, maintenant que le serveur tourne, on va pouvoir envoyer nos séries temporelles. Pour simplifier les choses, on va se concentrer sur la température.

In [8]:
#code client avec séries temporelles

import asyncio
import socket
import cbor2 as cbor
from virtual_sensor import virtual_sensor

humidity    = virtual_sensor(start=30, variation = 3, min=20, max=80)

s = socket.socket(socket.AF_INET, socket.SOCK_DGRAM)

async def client_minimal():
    NB_ELEMENT = 30
    h_history = []

    while True:
        if len(h_history) >= NB_ELEMENT:
            s.sendto (cbor.dumps(h_history), ("127.0.0.1", 33033))
            print(f"Envoyé : {h_history}")

            h_history = []

        h = int(humidity.read_value()*100) # in int * 100 to be compact 
        h_history.append(h)
        print (len(h_history), len(cbor.dumps(h_history)), h_history)

        
        await asyncio.sleep(2) # a small delay for not wainting too long

await client_minimal()

1 4 [2905]
2 7 [2905, 2726]
3 10 [2905, 2726, 2562]
4 13 [2905, 2726, 2562, 2536]
5 16 [2905, 2726, 2562, 2536, 2252]
6 19 [2905, 2726, 2562, 2536, 2252, 2421]
7 22 [2905, 2726, 2562, 2536, 2252, 2421, 2369]
8 25 [2905, 2726, 2562, 2536, 2252, 2421, 2369, 2138]
9 28 [2905, 2726, 2562, 2536, 2252, 2421, 2369, 2138, 2094]
10 31 [2905, 2726, 2562, 2536, 2252, 2421, 2369, 2138, 2094, 2232]
11 34 [2905, 2726, 2562, 2536, 2252, 2421, 2369, 2138, 2094, 2232, 2000]
12 37 [2905, 2726, 2562, 2536, 2252, 2421, 2369, 2138, 2094, 2232, 2000, 2000]
13 40 [2905, 2726, 2562, 2536, 2252, 2421, 2369, 2138, 2094, 2232, 2000, 2000, 2000]
14 43 [2905, 2726, 2562, 2536, 2252, 2421, 2369, 2138, 2094, 2232, 2000, 2000, 2000, 2000]
15 46 [2905, 2726, 2562, 2536, 2252, 2421, 2369, 2138, 2094, 2232, 2000, 2000, 2000, 2000, 2000]
16 49 [2905, 2726, 2562, 2536, 2252, 2421, 2369, 2138, 2094, 2232, 2000, 2000, 2000, 2000, 2000, 2000]
17 52 [2905, 2726, 2562, 2536, 2252, 2421, 2369, 2138, 2094, 2232, 2000, 2000, 2000

CancelledError: 

Pas mal, 77 octets pour 30 valeurs. Mais on peut faire mieux, en effet comme il s'agit d'un phénomène physique, les valeurs sont corrélées. Elles ne vont pas varier fortement d'une seconde à l'autre. Donc on peut coder la première valeur de manière absolue et les suivantes de manière relative. Comme CBOR n'utilise pas ma meme taille pour les entiers, on va pouvoir économiser un peu d'espace.

Stoppez le client précédent et lancez celui juste en dessous.

In [ ]:
#code client avec séries temporelles différentielles

import asyncio
import socket
import cbor2 as cbor
from virtual_sensor import virtual_sensor

humidity    = virtual_sensor(start=30, variation = 3, min=20, max=80)

s = socket.socket(socket.AF_INET, socket.SOCK_DGRAM)

async def client_minimal():
    NB_ELEMENT = 30
    h_history = []

    while True:
        if len(h_history) >= NB_ELEMENT:
            s.sendto (cbor.dumps(h_history), ("127.0.0.1", 33033))
            print(f"Envoyé : {h_history}")

            h_history = []

        h = int(humidity.read_value()*100) # in int * 100 to be compact 

        if len(h_history) == 0:
            h_history = [h]
        else:
            h_history.append(h-prev)
        print (len(h_history), len(cbor.dumps(h_history)), h_history)

        prev = h
        
        await asyncio.sleep(2) # a small delay for not wainting too long

await client_minimal()

1 4 [3071]
2 6 [3071, 184]
3 8 [3071, 184, -88]
4 10 [3071, 184, -88, 213]
5 11 [3071, 184, -88, 213, -7]
6 13 [3071, 184, -88, 213, -7, -47]
7 15 [3071, 184, -88, 213, -7, -47, -160]
8 17 [3071, 184, -88, 213, -7, -47, -160, -122]
9 19 [3071, 184, -88, 213, -7, -47, -160, -122, -211]
10 20 [3071, 184, -88, 213, -7, -47, -160, -122, -211, 20]
11 22 [3071, 184, -88, 213, -7, -47, -160, -122, -211, 20, 69]
12 24 [3071, 184, -88, 213, -7, -47, -160, -122, -211, 20, 69, 169]
13 26 [3071, 184, -88, 213, -7, -47, -160, -122, -211, 20, 69, 169, 174]
14 28 [3071, 184, -88, 213, -7, -47, -160, -122, -211, 20, 69, 169, 174, 48]
15 30 [3071, 184, -88, 213, -7, -47, -160, -122, -211, 20, 69, 169, 174, 48, 92]
16 32 [3071, 184, -88, 213, -7, -47, -160, -122, -211, 20, 69, 169, 174, 48, 92, -158]
17 35 [3071, 184, -88, 213, -7, -47, -160, -122, -211, 20, 69, 169, 174, 48, 92, -158, 292]
18 37 [3071, 184, -88, 213, -7, -47, -160, -122, -211, 20, 69, 169, 174, 48, 92, -158, 292, -131]
19 40 [3071, 184

La taille du message n'est plus systématiquement la même, mais on gagne une dizaine d'octets par message. Il faut évidemment modifier le serveur pour qu'il puisse gérer des séries temporelles différentielles.

In [ ]:
#code server
import cbor2 as cbor
import asyncio 

try:
    if not server_transport.is_closing():
        server_transport.close()
        print("🟡 Ancien serveur arrêté proprement.")
except NameError:
    # La variable n'existe pas encore (premier lancement)
    print("⚪ Premier lancement : aucun serveur à arrêter.")
    
# 1. Définition du protocole Serveur
class EchoServerProtocol(asyncio.DatagramProtocol):
    def connection_made(self, transport):
        self.transport = transport
        print("🟢 Serveur Asyncio UDP : DÉMARRÉ (127.0.0.1:33033)")

    def datagram_received(self, data, addr):
        data = cbor.loads(data)

        prev = 0
        for d in data:
            print(f"données: {d+prev} %HR")

# 2. Fonction pour lancer le serveur en arrière-plan
async def start_background_server():
    loop = asyncio.get_running_loop()
    
    # create_datagram_endpoint crée le socket et l'attache à la boucle
    transport, protocol = await loop.create_datagram_endpoint(
        lambda: EchoServerProtocol(),
        local_addr=('127.0.0.1', 33033)
    )
    
    # On retourne le transport pour pouvoir le fermer plus tard si besoin
    return transport

# 3. Lancement de la tâche serveur
# On stocke 'server_transport' dans une variable globale pour pouvoir l'arrêter
server_transport = await start_background_server()